In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sqrt
import warnings
warnings.filterwarnings("ignore")

## Underestanding Data

In [76]:
file_path = "movies.csv"
movies_df = pd.read_csv(file_path)
file_path2 = "ratings_sample.csv"
ratings_df = pd.read_csv(file_path2)
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [6]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,169,2.5,1204927694
1,1,2471,3.0,1204927438
2,1,48516,5.0,1204927435
3,2,2571,3.5,1436165433
4,2,109487,4.0,1436165496


## Preprocessing

In [9]:
movies_df['year'] = movies_df.title.str.extract("(\(\d\d\d\d\))", expand=False)
movies_df['year'] = movies_df.year.str.extract("(\d\d\d\d)", expand=False)
movies_df['title'] = movies_df.title.str.replace('(\(\d\d\d\d\))', '', regex=True)
movies_df['title'] = movies_df['title'].apply(lambda x: x.strip())
movies_df.head()

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995


In [11]:
movies_df['genres'] = movies_df.genres.str.split("|")
movies_df.head()

,movieId,title,genres,year
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",1995
1,2,Jumanji,"[Adventure, Children, Fantasy]",1995
2,3,Grumpier Old Men,"[Comedy, Romance]",1995
3,4,Waiting to Exhale,"[Comedy, Drama, Romance]",1995
4,5,Father of the Bride Part II,[Comedy],1995


In [13]:
movieswithgenres_df = movies_df.copy()

for index, row in movies_df.iterrows():
    for genre in row["genres"]:
        movieswithgenres_df.at[index, genre] = 1

movieswithgenres_df = movieswithgenres_df.fillna(0)
movieswithgenres_df.head()

,movieId,title,genres,year,Adventure,Animation,Children,Comedy,Fantasy,Romance,...,Horror,Mystery,Sci-Fi,IMAX,Documentary,War,Musical,Western,Film-Noir,(no genres listed)
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",1995,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,Jumanji,"[Adventure, Children, Fantasy]",1995,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,Grumpier Old Men,"[Comedy, Romance]",1995,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,Waiting to Exhale,"[Comedy, Drama, Romance]",1995,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,Father of the Bride Part II,[Comedy],1995,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
ratings_df = ratings_df.drop("timestamp", axis=1)
ratings_df.head()

,userId,movieId,rating
0,1,169,2.5
1,1,2471,3.0
2,1,48516,5.0
3,2,2571,3.5
4,2,109487,4.0


## contetnt_based Recommenders system

In [52]:
user_Id = 4
userInput = ratings_df[ratings_df['userId'] == user_Id]
inputMovies= pd.merge(userInput, movies_df, on='movieId')
inputMovies.head()

,userId,movieId,rating,title,genres,year
0,4,16,4.0,Casino,"[Crime, Drama]",1995
1,4,39,4.0,Clueless,"[Comedy, Romance]",1995
2,4,45,4.0,To Die For,"[Comedy, Drama, Thriller]",1995
3,4,47,2.0,Seven (a.k.a. Se7en),"[Mystery, Thriller]",1995
4,4,94,5.0,Beautiful Girls,"[Comedy, Drama, Romance]",1996


In [54]:
##inputId = movies_df[movies_df['title'].isin(inputMovies['title'].tolist())]
##inputMovies = pd.merge(inputId, inputMovies)
##inputMovies

In [56]:
inputMovies = inputMovies.drop('genres', axis=1).drop('year', axis=1)
inputMovies

,userId,movieId,rating,title
0,4,16,4.0,Casino
1,4,39,4.0,Clueless
2,4,45,4.0,To Die For
3,4,47,2.0,Seven (a.k.a. Se7en)
4,4,94,5.0,Beautiful Girls
...,...,...,...,...
178,4,7037,5.0,High Heels (Tacones lejanos)
179,4,7090,4.0,Hero (Ying xiong)
180,4,8464,4.5,Super Size Me
181,4,8622,5.0,Fahrenheit 9/11


In [58]:
userMovies = movieswithgenres_df[movieswithgenres_df['movieId'].isin(inputMovies['movieId'].tolist())]
userMovies

,movieId,title,genres,year,Adventure,Animation,Children,Comedy,Fantasy,Romance,...,Horror,Mystery,Sci-Fi,IMAX,Documentary,War,Musical,Western,Film-Noir,(no genres listed)
15,16,Casino,"[Crime, Drama]",1995,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
38,39,Clueless,"[Comedy, Romance]",1995,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
44,45,To Die For,"[Comedy, Drama, Thriller]",1995,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
46,47,Seven (a.k.a. Se7en),"[Mystery, Thriller]",1995,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
92,94,Beautiful Girls,"[Comedy, Drama, Romance]",1996,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6926,7037,High Heels (Tacones lejanos),"[Comedy, Drama]",1991,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6979,7090,Hero (Ying xiong),"[Action, Adventure, Drama]",2002,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7833,8464,Super Size Me,"[Comedy, Documentary, Drama]",2004,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
7940,8622,Fahrenheit 9/11,[Documentary],2004,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [60]:
userMovies = userMovies.reset_index(drop=True)
userGenresTabel = userMovies.drop('movieId', axis=1).drop('title', axis=1).drop('genres', axis=1).drop('year', axis=1)
userGenresTabel

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,IMAX,Documentary,War,Musical,Western,Film-Noir,(no genres listed)
0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
179,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
180,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
181,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [62]:
inputMovies['rating']

0      4.0
1      4.0
2      4.0
3      2.0
4      5.0
      ... 
178    5.0
179    4.0
180    4.5
181    5.0
182    4.0
Name: rating, Length: 183, dtype: float64

In [64]:
userProfile = userGenresTabel.transpose().dot(inputMovies['rating'])
userProfile

Adventure              56.0
Animation              25.0
Children               29.0
Comedy                384.5
Fantasy                33.0
Romance               212.0
Drama                 354.5
Action                 46.0
Crime                 107.0
Thriller              105.0
Horror                 20.0
Mystery                51.0
Sci-Fi                 16.0
IMAX                    0.0
Documentary            42.5
War                     5.0
Musical                26.0
Western                 3.0
Film-Noir               9.0
(no genres listed)      0.0
dtype: float64

In [66]:
genreTabel = movieswithgenres_df.set_index(movieswithgenres_df['movieId'])
genreTabel = genreTabel.drop('movieId', axis=1).drop('title', axis=1).drop('genres', axis=1).drop('year', axis=1)
genreTabel.head()

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,IMAX,Documentary,War,Musical,Western,Film-Noir,(no genres listed)
movieId,,,,,,,,,,,,,,,,,,,,
1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [68]:
genreTabel.shape

(34208, 20)

In [70]:
recommendationTabel_df = ((genreTabel*userProfile).sum(axis=1))/(userProfile.sum())
recommendationTabel_df.head()

movieId
1    0.346015
2    0.077402
3    0.391276
4    0.623811
5    0.252214
dtype: float64

In [72]:
recommendationTabel_df = recommendationTabel_df.sort_values(ascending=False)
recommendationTabel_df.head()

movieId
76153     0.842899
75408     0.842899
4719      0.809446
27781     0.809446
124681    0.799606
dtype: float64

In [74]:
movies_df.loc[movies_df['movieId'].isin(recommendationTabel_df.head(20).keys())]

,movieId,title,genres,year
953,970,Beat the Devil,"[Adventure, Comedy, Crime, Drama, Romance]",1953
1829,1912,Out of Sight,"[Comedy, Crime, Drama, Romance, Thriller]",1998
3801,3893,Nurse Betty,"[Comedy, Crime, Drama, Romance, Thriller]",2000
4625,4719,Osmosis Jones,"[Action, Animation, Comedy, Crime, Drama, Roma...",2001
4861,4956,"Stunt Man, The","[Action, Adventure, Comedy, Drama, Romance, Th...",1980
7511,7835,Song of the Thin Man,"[Comedy, Crime, Drama, Musical, Mystery, Romance]",1947
8605,26093,"Wonderful World of the Brothers Grimm, The","[Adventure, Animation, Children, Comedy, Drama...",1962
9296,27344,Revolutionary Girl Utena: Adolescence of Utena...,"[Action, Adventure, Animation, Comedy, Drama, ...",1999
9488,27781,Svidd Neger,"[Comedy, Crime, Drama, Horror, Mystery, Romanc...",2003
15001,75408,Lupin III: Sweet Lost Night (Rupan Sansei: Swe...,"[Action, Animation, Comedy, Crime, Drama, Myst...",2008
